# Building a Company Brochure with PDF Export using Gradio

## Imports & Dependencies

In [ ]:
import os
import json
from dotenv import load_dotenv
from urllib.parse import urljoin
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from xhtml2pdf import pisa
import gradio as gr

## Environment Setup & Model Configurations

In [ ]:
load_dotenv(override=True)

openrouter_api_key= os.getenv("OPENROUTER_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
ollama_api_key = "ollama"


openrouter_url = "https://openrouter.ai/api/v1"
gemini_url= "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url= "http://localhost:11434/v1"

gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key=ollama_api_key, base_url=ollama_url)


gemini_model = "gemini-2.5-flash"
gpt_model = "openai/gpt-4o-mini"
ollama_model = "llama3.2"

### System Prompt for Link Selection

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

### User Prompt Generator for Link Selection

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

### Select Relevant Web Links

In [ ]:
def select_relevant_links(url):
    
    response = openrouter.chat.completions.create(
        model=gemini_model,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    
    return links

### Aggregate Landing Page & Sub-Page Contents

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

### System Prompt for Brochure Generation

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of a company website landing page and sub-pages
and creates a short, engaging brochure about the company for prospective customers, investors, and recruits.
Respond in markdown without code blocks.
"""

### Build User Prompt for Brochure

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:15_000] # Truncate if more than 5,000 characters
    return user_prompt

### Non-Streaming Brochure Generation

In [ ]:
def create_brochure(company_name, url):
    print(f"Creating Brochure for  {company_name} by calling {gemini_model}")
    response = openrouter.chat.completions.create(
        model=gemini_model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    
    display(Markdown(result))
    return result

## Streaming LLM Generation Functions

In [ ]:
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": prompt}
    ]
    stream = openrouter.chat.completions.create(
        model=gpt_model,
        messages=messages,
        stream=True,
        max_tokens=2000  # Prevents OpenRouter 402 credit error
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
def stream_gemini(prompt):
    messages = [
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": prompt}
    ]
    stream = gemini.chat.completions.create(
        model=gemini_model,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

## PDF Conversion from Brochure Markdown

In [ ]:
html_system_prompt = """
You are an expert document designer for PDF conversion.
Convert the provided Markdown brochure into a clean, standalone HTML document for xhtml2pdf.
CRITICAL PDF ENGINE RULES:
1. DO NOT use <table> tags. Represent lists and structured data using <ul>, <li>, <div>, and heading tags (<h1>, <h2>, <h3>).
2. DO NOT use CSS variables (like var(...)). Use direct HEX color codes (e.g., #1e40af, #1e293b, #f8fafc, #333333).
3. DO NOT use @font-face or external font imports.
4. Include @page { size: letter; margin: 0.75in; } in the <style> block.
5. Return ONLY the raw HTML string inside <html>...</html> without markdown backticks.
"""


In [ ]:
def convert_brochure_to_pdf(brochure_markdown, company_name):
    # Sanitize filename (e.g., "Hugging Face" -> "hugging_face_brochure.pdf")
    safe_name = "".join([c for c in company_name if c.isalnum() or c in (' ', '_')]).strip().replace(" ", "_").lower()
    pdf_filename = f"{safe_name}_brochure.pdf"
    
    print(f"Converting brochure for {company_name} to styled HTML...")
    
    html_response = gemini.chat.completions.create(
        model=gemini_model,  # Using defined model
        messages=[
            {"role": "system", "content": html_system_prompt},
            {"role": "user", "content": f"Company: {company_name}\n\nBrochure:\n{brochure_markdown}"}
        ],
    )
    
    html_content = html_response.choices[0].message.content
    
    # Clean any ```html backticks from LLM output
    if "```" in html_content:
        html_content = html_content.split("```html")[-1].split("```")[0].strip()
        
    print(f"Saving PDF to {pdf_filename}...")
    with open(pdf_filename, "wb") as f:
        pisa_status = pisa.CreatePDF(html_content, dest=f)
        
    if pisa_status.err:
        print("Warning: PDF rendered with minor layout warnings.")
    else:
        print(f"Successfully created PDF: {pdf_filename}")
        
    return pdf_filename

## Main Gradio Handler Function

In [ ]:
def stream_brochure(company_name, url, model, generate_pdf):
    # Step 1: Initial user feedback
    yield ("🔍 **Fetching landing page and analyzing relevant sub-pages...**", None)
    
    # Step 2: Generate the context prompt
    prompt = get_brochure_user_prompt(company_name, url)
    
    # Step 3: Stream the brochure content
    stream_fn = stream_gpt if model == "GPT" else stream_gemini
    full_markdown = ""
    
    for chunk in stream_fn(prompt):
        full_markdown = chunk
        # Yield (markdown_text, file_path) -> None for file while streaming text
        yield (full_markdown, None)
        
    # Step 4: If PDF checkbox is enabled, create PDF at the end
    if generate_pdf and full_markdown:
        yield (full_markdown + "\n\n📄 *Converting brochure to downloadable PDF...*", None)
        pdf_path = convert_brochure_to_pdf(full_markdown, company_name)
        # Final yield with the generated PDF file path
        yield (full_markdown, pdf_path)

## Launch Gradio Interactive Dashboard

In [ ]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "Gemini"], label="Select model", value="GPT")
pdf_checkbox = gr.Checkbox(label="Generate Downloadable PDF", value=False)
message_output = gr.Markdown(label="Brochure Response:")
pdf_download = gr.File(label="Download PDF Brochure")
view = gr.Interface(
    fn=stream_brochure,
    title="Brochure & PDF Generator", 
    inputs=[name_input, url_input, model_selector, pdf_checkbox], 
    outputs=[message_output, pdf_download], 
    examples=[
        ["Hugging Face", "https://huggingface.co", "GPT", True],
        ["Edward Donner", "https://edwarddonner.com", "Gemini", False]
    ], 
    flagging_mode="never"
)
view.launch()